## Utilizando Web-Scraping para levantar a literatura

#### Passo 1: instalar bibliotecas

In [37]:
# Remova o comentário(#) da linha abaixo caso esteja executando este script pela primeira vez
#!pip install requests
#!pip install beautifulsoup4

#### Passo 2: Importar bibliotecas

In [38]:
import requests
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

#### Passo 3: Formatar link de busca

In [39]:
# Copie o modelo do link de busca
url_busca = 'https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html?source=all&q='


# Informe a palvra chave
palavra_chave = 'pokemon'


# Formata link
link = url_busca+palavra_chave
print(link)

https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html?source=all&q=pokemon


#### Passo 4: Obter informações do link

In [40]:
# Requisita informações do site
response = requests.get(link, timeout=10)


# Verifica status de acesso
print(response.status_code)


403


Esse **403 Forbidden** acontece porque o Portal da Capes possui mecanismos de proteção contra bots e scrapers simples, e o requests.get() sem nenhum cabeçalho já denuncia que não é um navegador “de verdade”.

#### 4.1 Usar um User-Agent válido

Alguns sites **bloqueiam requisições que não parecem vir de navegadores**. Então você pode **enviar cabeçalhos simulando um browser**.

In [41]:
# Cabeçalho com User-Agent
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.174 Safari/537.3"}

# Requisita informações do site
response = requests.get(link, headers=headers, timeout=10)

# Verifica status de acesso
print(response.status_code)
print("Content-Type: ", response.headers.get("Content-Type"))



200
Content-Type:  text/html; charset=utf-8


#### Passo 5: Encontrar elementos

In [42]:
soup = BeautifulSoup(response.text, 'html.parser')

In [43]:
print(soup.prettify()[:500])

<!DOCTYPE html>
<html lang="pt-br" xml:lang="pt-br" xmlns="http://www.w3.org/1999/xhtml">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
  <meta content="ie=edge" http-equiv="X-UA-Compatible">
   <meta content="#00a300" name="theme-color"/>
   <meta content="https://www.periodicos.capes.gov.br/templates/periodicos_gov/images/icon-periodicos.png" property="og:image"/>
   <meta content="Portal de Periódicos da CAPES" prop


In [44]:
print("Titulo da pagina: ", soup.title.get_text(strip=True))

Titulo da pagina:  Buscar assunto


In [45]:
# Encontrar por id
# Pegando primeiro artigo
artigo = soup.find("div", id="conteudo-0") # Primeiro artigo
print(artigo.prettify())


<div class="col-md-12 br-item" id="conteudo-0">
 <div class="row">
  <div class="d-flex align-items-center text-uppercase text-down-01 px-0">
   <div class="br-checkbox hidden-label small mr-2">
    <input aria-label="0" data-child="check-itens" id="checkbox-0" name="checkbox-0" type="checkbox"/>
    <label for="checkbox-0">
     0
    </label>
   </div>
   <div class="br-checkbox br-checkbox-item">
    <label for="check-0">
     0
    </label>
    <input id="check-0" name="itens" type="checkbox"/>
   </div>
   <span class="fw-semibold">
    Livro
   </span>
  </div>
 </div>
 <div class="row mb-2">
  <div class="col-md-12">
   <div class="d-flex py-2 m-0 text-up-02 text-weight-semi-bold">
    <span class="text-muted">
     <strong>
      1.
     </strong>
    </span>
    <a class="titulo-busca" href="/index.php/acervo/buscador.html?task=detalhes&amp;source=resources&amp;id=WB3240000000019956" type="button">
     Public Versus Secret Reserve Prices in eBay Auctions: Results from a
     

In [46]:
#Título
link_titulo = artigo.find("a", class_="titulo-busca")
titulo = link_titulo.get_text().strip()
print(titulo)

Public Versus Secret Reserve Prices in eBay Auctions: Results from a Pokemon Field Experiment


#### Passo 6: Armazenar dados

In [47]:
# Cria uma lista vazia para guardar dados
lista = []
artigos = soup.find_all("div", id=re.compile("conteudo-")) # Todos artigo(re.compile)


# Cria laço de repetição para filtrar todos os dados
for artigo_atual in artigos:
    
    link_titulo = artigo_atual.find("a", class_="titulo-busca")
    titulo = link_titulo.get_text().strip()
    
    artigo = {
        'link': urljoin(url_busca, link_titulo.get('href')),
        'titulo': titulo
    }
    
    lista.append(artigo)
    

#### Passo 7: Visualizar listas

In [48]:
print(lista)

[{'link': 'https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html?task=detalhes&source=resources&id=WB3240000000019956', 'titulo': 'Public Versus Secret Reserve Prices in eBay Auctions: Results from a Pokemon Field Experiment'}, {'link': 'https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html?task=detalhes&source=digital&id=W4297036146', 'titulo': 'Augmented Pokemon Go in Times of Covid-19: Does it Have any Effect on Promoting Teenagers’ Physical Activity?'}, {'link': 'https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html?task=detalhes&source=resources&id=WB4920000000464592', 'titulo': 'Death by Pokémon GO : The Economic and Human Cost of Using Apps While Driving'}, {'link': 'https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html?task=detalhes&source=digital&id=W2917069409', 'titulo': 'Pokemon Inhibits Transforming Growth Factor β-Smad4-Related Cell Proliferation Arrest in Breast Cancer through Specificity Protein 1'}, {'link': 'https://

#### Passo 8: Organizar tabelas e salvar

In [49]:
# Importar biblioteca Pandas para criar DataFrames
import pandas as pd


In [50]:
# Transforma arrays em DataFrame
# Salvar DataFrame
df = pd.DataFrame(lista )


In [51]:
# Visualizar DataFrame
df.head()

,link,titulo
0,https://www.periodicos.capes.gov.br/index.php/...,Public Versus Secret Reserve Prices in eBay Au...
1,https://www.periodicos.capes.gov.br/index.php/...,Augmented Pokemon Go in Times of Covid-19: Doe...
2,https://www.periodicos.capes.gov.br/index.php/...,Death by Pokémon GO : The Economic and Human C...
3,https://www.periodicos.capes.gov.br/index.php/...,Pokemon Inhibits Transforming Growth Factor β-...
4,https://www.periodicos.capes.gov.br/index.php/...,Gotta catch ‘em all or not enough time: Users ...


#### Exercício: Extrair as demais informações abaixo
- Título
- Autores
- Editora
- Volume
- Edição (Número)
- Idioma
- Ano
- Tópicos
- DOI
- ISSN
- Resumo (sobre)
- Link

#### Tentativa de resposta

In [52]:
dados = []

for artigo_atual in artigos:
    
    
    link_titulo = artigo_atual.find("a", class_="titulo-busca")
    titulo = link_titulo.get_text().strip()
        
    
    link = urljoin(url_busca, link_titulo.get("href")) # Pega o link da pagina
    
    dicionario = {
        'titulo': titulo,
        'autores': "Não informado",
        'editora': "Não informado",
        'volume': "Não informado",
        'edicao': "Não informado",
        'idioma': "Não informado",
        'ano': "Não informado",
        'topicos': "Não informado",
        'DOI': "Não informado",
        'ISSN': "Não informado",
        'resumo': "Não informado",
        'link': link
    } 
    
    artigo_completo = requests.get(link, headers=headers, timeout=10) # Entra na pagina
    artigo = BeautifulSoup(artigo_completo.text, 'html.parser') # Pega o html

    tabela = artigo.find('table')
    

    for linha in tabela.find_all('tr'):
        coluna = linha.find_all(['th', 'td'])
        
        if len(coluna) >= 2:
            chave = coluna[0].get_text(strip=True).lower()
            valor = " ".join(coluna[1].get_text().split())
            
            if 'autores' in chave:
                dicionario['autores'] = valor
            elif 'editora' in chave:
                dicionario['editora'] = valor
            elif 'volume' in chave:
                dicionario['volume'] = valor
            elif 'edição' in chave or 'número' in chave:
                dicionario['edicao'] = valor
            elif 'idioma' in chave:
                dicionario['idioma'] = valor
            elif 'ano' in chave or 'data' in chave:
                dicionario['ano'] = valor
            elif 'tópico' in chave or 'assunto' in chave:
                dicionario['topicos'] = valor
            elif 'doi' in chave:
                dicionario['DOI'] = valor
            elif 'issn' in chave:
                dicionario['ISSN'] = valor
                
    resumo = artigo.find("p", id="item-resumo")
    if resumo:
        dicionario['resumo'] = " ".join(resumo.get_text().split())


        
    dados.append(dicionario)

In [53]:
for um in dados:
    print(um)
    print("\n")

{'titulo': 'Public Versus Secret Reserve Prices in eBay Auctions: Results from a Pokemon Field Experiment', 'autores': 'Não informado', 'editora': 'National Bureau of Economic Research (NBER)', 'volume': 'Não informado', 'edicao': 'Não informado', 'idioma': 'Português', 'ano': 'Não informado', 'topicos': 'Não informado', 'DOI': 'Não informado', 'ISSN': 'Não informado', 'resumo': 'Não informado', 'link': 'https://www.periodicos.capes.gov.br/index.php/acervo/buscador.html?task=detalhes&source=resources&id=WB3240000000019956'}


{'titulo': 'Augmented Pokemon Go in Times of Covid-19: Does it Have any Effect on Promoting Teenagers’ Physical Activity?', 'autores': 'Hasanuddin Jumareng, Edi Setiawan, Zsolt Németh,', 'editora': 'OVS LLC', 'volume': '22', 'edicao': '3', 'idioma': 'Inglês', 'ano': '2022', 'topicos': 'Obesity, Physical Activity, Diet', 'DOI': '10.17309/tmfv.2022.3.09', 'ISSN': '1993-7997', 'resumo': 'Study purpose. One of the main problems during the COVID-19 pandemic that needs 

In [54]:
df_artigos = pd.DataFrame(dados)

In [55]:
df_artigos.head()

,titulo,autores,editora,volume,edicao,idioma,ano,topicos,DOI,ISSN,resumo,link
0,Public Versus Secret Reserve Prices in eBay Au...,Não informado,National Bureau of Economic Research (NBER),Não informado,Não informado,Português,Não informado,Não informado,Não informado,Não informado,Não informado,https://www.periodicos.capes.gov.br/index.php/...
1,Augmented Pokemon Go in Times of Covid-19: Doe...,"Hasanuddin Jumareng, Edi Setiawan, Zsolt Németh,",OVS LLC,22,3,Inglês,2022,"Obesity, Physical Activity, Diet",10.17309/tmfv.2022.3.09,1993-7997,Study purpose. One of the main problems during...,https://www.periodicos.capes.gov.br/index.php/...
2,Death by Pokémon GO : The Economic and Human C...,Não informado,National Bureau of Economic Research (NBER),Não informado,Não informado,Português,Não informado,Não informado,Não informado,Não informado,Não informado,https://www.periodicos.capes.gov.br/index.php/...
3,Pokemon Inhibits Transforming Growth Factor β-...,"Ling Chen, Jing Zhong, Jiang-Hua Liu, Duan-Fan...",Não informado,22,1,Inglês,2019,Cytokine Signaling Pathways and Interactions,10.4048/jbc.2019.22.e11,2092-9900,"Pokemon, also known as ZBTB7A, belongs to the ...",https://www.periodicos.capes.gov.br/index.php/...
4,Gotta catch ‘em all or not enough time: Users ...,"David Robert Broom, Ka Yiu Lee, Michael Huen S...",PAGEPress (Italy),7,1,Inglês,2019,"Obesity, Physical Activity, Diet",10.4081/hpr.2019.7714,2420-8124,Urban exergames are played in the real-world e...,https://www.periodicos.capes.gov.br/index.php/...
